# 03 — Build RAG Index

This notebook produces the **precomputed retrieval artifacts** that Star
Engine's browser-side RAG pipeline consumes:

| File | What it is | ~Size |
|---|---|---|
| `public/data/index/chunks/manifest.json` + `chunks/{0000..0009}.jsonl` | Section-aware chunks of every paper's full text, sharded by `hash(pmc_id) % 10` | ~10 MB |
| `public/data/index/embeddings.f16.bin` + `embeddings.meta.json` | MiniLM-L6-v2 chunk embeddings, 384-d float16 | ~5 MB |
| `public/data/index/bm25.json` | Sparse postings list, df table, avgdl | ~2–4 MB |
| `public/data/index/kg_node_embeddings.f16.bin` + `kg_node_meta.json` | MiniLM embeddings of `{name}. {provenance}` for entity-linking concept nodes | ~2 MB |
| `public/data/kg_lean.json` | Runtime-sized KG (no `provenance` text) | ~1.5 MB |

## Prerequisites

- The 494 paper folders **on disk** at `data/papers/{pmc_id}/full_text.xml`
  (produced once by `notebooks/00_legacy_pipeline.ipynb`). These are
  gitignored — the runtime app never needs them.
- `public/data/publications.json` and `public/data/knowledge_graph.json`
  (already in the repo).

## Where to run

Anywhere you have Python ≥ 3.10 and can install pip packages. The
notebook assumes the repo is checked out locally; the next cell
auto-detects the project root from the marker files. If you're using a
remote kernel (Colab, JupyterHub, etc.) and the kernel can't see your
local files, mount your repo into the kernel's filesystem (Drive, SSH
sync, etc.) and set `PROJECT_ROOT_OVERRIDE` to wherever it landed.

## Runtime

- ~3-10 min total, dominated by the MiniLM embedding pass.
- A CUDA GPU brings the embedding step from ~5 min down to under a minute
  — sentence-transformers picks it up automatically if available.

## What this notebook does NOT do

- It does **not** call any LLM. All embeddings are computed locally
  with the same open-weights MiniLM the browser uses at query time, so
  precompute and runtime are perfectly consistent.
- It does **not** redo the Gemini knowledge-graph extraction. That's
  `notebooks/00_legacy_pipeline.ipynb` (already run).


## 1. Setup — paths and dependencies

In [1]:
import os, sys, json, re, hashlib, struct, math, time
from pathlib import Path

# --- Project root ---
# Set this if auto-detect doesn't land on the right folder. The string is
# raw (r'...') so backslashes in Windows paths don't need escaping.
PROJECT_ROOT_OVERRIDE = None
# Example for Moaz's machine:
#   PROJECT_ROOT_OVERRIDE = Path(r'C:\Users\moaz\Desktop\APPs\StarEngine')

# Files we expect to find at the project root.
PROJECT_MARKERS = [
    Path('public') / 'data' / 'publications.json',
    Path('public') / 'data' / 'knowledge_graph.json',
    Path('package.json'),
]

def _has_markers(p: Path) -> bool:
    return all((p / m).exists() for m in PROJECT_MARKERS)

def _find_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE is not None:
        p = Path(PROJECT_ROOT_OVERRIDE).resolve()
        if not _has_markers(p):
            print(f"WARNING: PROJECT_ROOT_OVERRIDE={p} doesn't have the expected files.")
        return p

    # Walk up from cwd AND (if the IDE injected it) from the notebook's
    # own file location. Notebooks often run with cwd=$HOME or even /,
    # so cwd alone is unreliable.
    candidates = []
    candidates.append(Path.cwd().resolve())
    candidates.extend(Path.cwd().resolve().parents)
    try:
        ipy = get_ipython()
        nb_path = ipy.user_ns.get('__vsc_ipynb_file__') or ipy.user_ns.get('__session__')
        if nb_path:
            nb = Path(nb_path).resolve()
            candidates.append(nb.parent)
            candidates.extend(nb.parents)
    except Exception:
        pass

    seen = set()
    for c in candidates:
        c = c.resolve()
        if c in seen:
            continue
        seen.add(c)
        if _has_markers(c):
            return c

    raise RuntimeError(
        "Could not auto-detect the StarEngine project root.\n"
        f"  cwd was: {Path.cwd()}\n"
        "Fix: set PROJECT_ROOT_OVERRIDE at the top of this cell, e.g.\n"
        "  PROJECT_ROOT_OVERRIDE = Path(r'C:\\Users\\moaz\\Desktop\\APPs\\StarEngine')\n"
        "Then re-run this cell."
    )

PROJECT_ROOT  = _find_project_root()
PUBLIC_DIR    = PROJECT_ROOT / 'public'
PAPERS_DIR    = PROJECT_ROOT / 'data' / 'papers'
PUB_JSON      = PUBLIC_DIR / 'data' / 'publications.json'
KG_FULL_JSON  = PUBLIC_DIR / 'data' / 'knowledge_graph.json'
INDEX_DIR     = PUBLIC_DIR / 'data' / 'index'
CHUNKS_DIR    = INDEX_DIR / 'chunks'
KG_LEAN_JSON  = PUBLIC_DIR / 'data' / 'kg_lean.json'

INDEX_DIR.mkdir(parents=True, exist_ok=True)
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"PUBLIC_DIR:   {PUBLIC_DIR}")
print(f"PAPERS_DIR:   {PAPERS_DIR}   (exists: {PAPERS_DIR.exists()})")
print(f"PUB_JSON:     {PUB_JSON}     (exists: {PUB_JSON.exists()})")
print(f"KG_FULL:      {KG_FULL_JSON} (exists: {KG_FULL_JSON.exists()})")
print(f"INDEX_DIR:    {INDEX_DIR}")

# Hard-fail if essential inputs are missing — better than scribbling
# empty artifacts and confusing the runtime later.
missing = [str(p) for p in (PUB_JSON, KG_FULL_JSON) if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Required input files are missing:\n  " + "\n  ".join(missing) +
        "\n\nMake sure the repo is fully cloned (public/data/ included) and "
        "that notebook 00 has been run to produce publications.json + "
        "knowledge_graph.json."
    )
if not PAPERS_DIR.exists() or not any(PAPERS_DIR.iterdir()):
    print(
        "\nNOTE: data/papers/ is empty or missing. The chunking step (cell 5) "
        "needs the 494 paper XMLs from notebook 00 there. You can still run "
        "the KG-node embedding + kg_lean steps without it."
    )

PROJECT_ROOT: C:\Users\moaz\Desktop\APPs\StarEngine
PUBLIC_DIR:   C:\Users\moaz\Desktop\APPs\StarEngine\public
PAPERS_DIR:   C:\Users\moaz\Desktop\APPs\StarEngine\data\papers   (exists: True)
PUB_JSON:     C:\Users\moaz\Desktop\APPs\StarEngine\public\data\publications.json     (exists: True)
KG_FULL:      C:\Users\moaz\Desktop\APPs\StarEngine\public\data\knowledge_graph.json (exists: True)
INDEX_DIR:    C:\Users\moaz\Desktop\APPs\StarEngine\public\data\index


In [2]:
# Install dependencies (idempotent — pip skips already-installed packages).
%pip install -q sentence-transformers==2.7.0 rank-bm25==0.2.2 lxml==5.* nltk==3.* numpy
import nltk
for pkg in ('punkt', 'punkt_tab', 'stopwords'):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass
print('Dependencies ready.')


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Dependencies ready.


## 2. Section-aware chunking

The raw papers are JATS XML. Naive chunking (split every N chars) would
cut across section boundaries and lose context. Instead:

1. Walk top-level sections (`<sec>`/`<title>`/`<p>`) plus the abstract.
2. If a section fits in `TARGET_TOKENS`, it becomes one chunk.
3. Otherwise, split on sentence boundaries with `OVERLAP_TOKENS` of
   carry-over so retrieval at the boundary still finds the right answer.

Each chunk records `(chunk_id, pmc_id, section, n_tokens, text)`. We
shard chunks by `hash(pmc_id) % 10` into ten JSONL files so the browser
can HTTP-Range-fetch a single paper's chunks without downloading the
whole corpus.

In [ ]:
import lxml.etree as ET
from nltk.tokenize import sent_tokenize

TARGET_TOKENS  = 600
OVERLAP_TOKENS = 100

def _strip_namespaces(tree):
    """In-place rewrite of every tag to drop its '{ns}' prefix.

    JATS XML comes in several namespace flavors (the legacy notebook 00
    has a 3-stage fallback for this). Stripping once at parse time lets
    the rest of the code use plain '.find(".//sec")' instead of building
    a per-document namespace map. lxml's simplified XPath supported by
    Element.find() doesn't accept local-name() predicates, so this is
    actually the cleanest path.
    """
    for elem in tree.iter():
        if isinstance(elem.tag, str) and '}' in elem.tag:
            elem.tag = elem.tag.split('}', 1)[1]
        # Also strip namespace prefixes from attributes
        if elem.attrib:
            new_attrib = {}
            for k, v in elem.attrib.items():
                new_attrib[k.split('}', 1)[1] if '}' in k else k] = v
            elem.attrib.clear()
            elem.attrib.update(new_attrib)
    return tree

def _text(el):
    return re.sub(r"\s+", " ", ''.join(el.itertext())).strip() if el is not None else ''

def _count_tokens(s):
    return len(re.findall(r"\w+", s))

def _pack_with_overlap(text, target=TARGET_TOKENS, overlap=OVERLAP_TOKENS):
    sentences = sent_tokenize(text)
    chunks, current, cur_tokens = [], [], 0
    for sent in sentences:
        st = _count_tokens(sent)
        if cur_tokens + st > target and current:
            chunks.append(' '.join(current))
            ovl, ot = [], 0
            for s in reversed(current):
                if ot + _count_tokens(s) > overlap:
                    break
                ovl.insert(0, s)
                ot += _count_tokens(s)
            current, cur_tokens = ovl, ot
        current.append(sent)
        cur_tokens += st
    if current:
        chunks.append(' '.join(current))
    return chunks

def chunks_for_paper(pmc_id, xml_path):
    """Return list[dict] of chunks for one paper, or [] on parse failure."""
    try:
        tree = ET.parse(str(xml_path))
    except Exception:
        return []
    root = _strip_namespaces(tree).getroot()

    # Collect sections: abstract + top-level <sec> in <body>.
    sections = []
    abst = root.find('.//abstract')
    if abst is not None:
        sections.append(('Abstract', _text(abst)))
    body = root.find('.//body')
    if body is not None:
        for sec in body.findall('./sec'):
            title_el = sec.find('title')
            title = (title_el.text or 'Section').strip() if title_el is not None and title_el.text else 'Section'
            body_text = _text(sec)
            if title_el is not None and title_el.text:
                # Drop the leading title text from the body string so it
                # doesn't appear twice in the chunk.
                body_text = body_text.replace(title_el.text.strip(), '', 1).strip()
            if body_text:
                sections.append((title, body_text))

    out = []
    for sec_title, sec_text in sections:
        if not sec_text:
            continue
        pieces = [sec_text] if _count_tokens(sec_text) <= TARGET_TOKENS else _pack_with_overlap(sec_text)
        for i, piece in enumerate(pieces):
            out.append({
                'chunk_id': f'{pmc_id}:{sec_title}:{i}',
                'pmc_id': pmc_id,
                'section': sec_title,
                'n_tokens': _count_tokens(piece),
                'text': piece,
            })
    return out

# --- Smoke test on one paper ---
sample_dirs = sorted([d for d in PAPERS_DIR.iterdir() if d.is_dir()])
if sample_dirs:
    s = sample_dirs[0]
    xml_path = s / 'full_text.xml'
    if xml_path.exists():
        sample = chunks_for_paper(s.name, xml_path)
        print(f'Sample {s.name}: {len(sample)} chunks ({sum(c["n_tokens"] for c in sample)} tokens total)')
        if sample:
            print('First chunk:', sample[0]['section'], '→', sample[0]['text'][:200], '…')
    else:
        print(f'WARNING: {xml_path} not found. Did you run notebook 00 first?')
else:
    print('WARNING: no paper directories found in', PAPERS_DIR)


In [ ]:
from tqdm.auto import tqdm

# Run the chunker over every paper, then shard the result.
SHARDS = 10
shard_buckets = [[] for _ in range(SHARDS)]
manifest = {}  # pmc_id -> { shard, byte_offset, byte_length }

all_paper_dirs = sorted([d for d in PAPERS_DIR.iterdir() if d.is_dir()])
skipped = []

for d in tqdm(all_paper_dirs, desc='Chunking papers'):
    xml_path = d / 'full_text.xml'
    if not xml_path.exists():
        skipped.append(d.name)
        continue
    chunks = chunks_for_paper(d.name, xml_path)
    if not chunks:
        skipped.append(d.name)
        continue
    shard = int(hashlib.md5(d.name.encode()).hexdigest(), 16) % SHARDS
    shard_buckets[shard].append((d.name, chunks))

# Write shards, capturing byte ranges per paper so the browser can
# HTTP-Range-fetch just the chunks for ONE paper without pulling the
# full shard.
total_chunks = 0
for shard_idx, bucket in enumerate(shard_buckets):
    out_path = CHUNKS_DIR / f'{shard_idx:04d}.jsonl'
    cursor = 0
    with open(out_path, 'wb') as f:
        for pmc_id, chunks in bucket:
            start = cursor
            for ch in chunks:
                line = (json.dumps(ch, ensure_ascii=False) + '\n').encode('utf-8')
                f.write(line)
                cursor += len(line)
                total_chunks += 1
            manifest[pmc_id] = {
                'shard': shard_idx,
                'byte_offset': start,
                'byte_length': cursor - start,
                'n_chunks': len(chunks),
            }

with open(CHUNKS_DIR / 'manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, separators=(',', ':'))

print(f'Chunked {len(manifest)} papers, {total_chunks} chunks total.')
print(f'Skipped {len(skipped)} papers (no XML or parse failure).')
for shard_idx in range(SHARDS):
    sz = (CHUNKS_DIR / f'{shard_idx:04d}.jsonl').stat().st_size
    print(f'  shard {shard_idx:04d}: {sz/1024:.1f} KB')

## 3. MiniLM chunk embeddings

We embed every chunk with `sentence-transformers/all-MiniLM-L6-v2`:

- 384-d, fp16 — the smallest defensible dense-retrieval baseline.
- Open weights, runs in the browser via `@xenova/transformers` at
  query time → precompute and runtime are perfectly consistent.
- Stored as a single fp16 binary `embeddings.f16.bin` (concatenated
  rows). Sidecar `embeddings.meta.json` maps `chunk_id -> row_index`.
- A separate Float32Array view at runtime gives ~10 ms cosine over
  ~12k vectors — no ANN structure needed at this scale.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
BATCH_SIZE = 64

# Re-load every chunk from the sharded JSONL into memory (modest size,
# ~30 MB). chunk_ids preserves order = embedding row order.
all_chunks = []
for shard_idx in range(SHARDS):
    with open(CHUNKS_DIR / f'{shard_idx:04d}.jsonl', 'r', encoding='utf-8') as f:
        for line in f:
            all_chunks.append(json.loads(line))
print(f'Loaded {len(all_chunks)} chunks for embedding.')

model = SentenceTransformer(MODEL_NAME)
device = 'cuda' if hasattr(model, '_target_device') and 'cuda' in str(model._target_device) else 'cpu'
print(f'Embedding on {device} …')

texts = [c['text'] for c in all_chunks]
embs = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # so we can use plain dot product at runtime
).astype(np.float16)

print('Embeddings shape:', embs.shape, 'dtype:', embs.dtype)

# Persist as binary + sidecar meta
bin_path  = INDEX_DIR / 'embeddings.f16.bin'
meta_path = INDEX_DIR / 'embeddings.meta.json'
with open(bin_path, 'wb') as f:
    f.write(embs.tobytes())

with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump({
        'model': MODEL_NAME,
        'dim': int(embs.shape[1]),
        'rows': int(embs.shape[0]),
        'dtype': 'float16',
        'normalized': True,
        'chunk_ids': [c['chunk_id'] for c in all_chunks],
    }, f, ensure_ascii=False, separators=(',', ':'))

print(f'Wrote {bin_path} ({bin_path.stat().st_size/1024/1024:.2f} MB).')
print(f'Wrote {meta_path} ({meta_path.stat().st_size/1024:.1f} KB).')

## 4. BM25 sparse index

Hybrid retrieval needs sparse scoring alongside dense. We use textbook
BM25 (`k1=1.5, b=0.75`) with Porter stems and English stopwords. The
browser does scoring directly against this JSON — no extra deps.

In [ ]:
from nltk.corpus import stopwords as nltk_stopwords
from collections import Counter, defaultdict

STOPS = set(nltk_stopwords.words('english'))
TOKEN_RE = re.compile(r"[a-z0-9]+")

# Tokenizer: identical Python/JS behavior so src/rag/tokenizer.js in the
# browser matches what we precomputed here. We intentionally skip stemming
# — with MiniLM dense retrieval doing the heavy lifting, Porter stemming's
# Recall@10 lift is small (~3-5%) and not worth the precompute/runtime-
# drift risk of two stemmer implementations getting out of sync. The exact
# stopword list ships INSIDE bm25.json so the JS query tokenizer reuses it.
def tokenize(text):
    return [t for t in TOKEN_RE.findall(text.lower()) if t not in STOPS and len(t) > 1]

# Pass 1: per-chunk token frequencies, doc lengths, document frequency.
doc_lens = []
term_doc_freq = Counter()    # term -> number of docs containing term
tf_per_doc = []              # list of Counter(term -> tf) aligned with chunk row

for ch in tqdm(all_chunks, desc='Tokenizing for BM25'):
    toks = tokenize(ch['text'])
    tf = Counter(toks)
    tf_per_doc.append(tf)
    doc_lens.append(len(toks))
    for term in tf.keys():
        term_doc_freq[term] += 1

N = len(all_chunks)
avgdl = sum(doc_lens) / max(N, 1)
print(f'BM25 stats: N={N}, avgdl={avgdl:.1f}, vocab={len(term_doc_freq)}')

# Build inverted postings: term -> list of [doc_index, tf].
postings = defaultdict(list)
for doc_idx, tf in enumerate(tf_per_doc):
    for term, freq in tf.items():
        postings[term].append([doc_idx, freq])

# Drop ultra-rare terms (df=1) — they bloat the JSON without retrieval
# value at our corpus size. Comment out if you want maximum recall.
before = len(postings)
postings = {t: p for t, p in postings.items() if len(p) > 1}
print(f'Pruned {before - len(postings)} singleton terms; final vocab {len(postings)}.')

bm25 = {
    'k1': 1.5,
    'b': 0.75,
    'N': N,
    'avgdl': avgdl,
    'doc_lens': doc_lens,
    'stops': sorted(STOPS),        # ship the exact stopword list to the browser
    'df': {t: len(p) for t, p in postings.items()},
    'postings': postings,
    'chunk_ids': [c['chunk_id'] for c in all_chunks],
}

bm25_path = INDEX_DIR / 'bm25.json'
with open(bm25_path, 'w', encoding='utf-8') as f:
    json.dump(bm25, f, ensure_ascii=False, separators=(',', ':'))
print(f'Wrote {bm25_path} ({bm25_path.stat().st_size/1024/1024:.2f} MB).')


## 5. GraphRAG: KG-node embeddings + lean KG

Two outputs here:

1. **KG-node embeddings** — used for entity-linking the query at runtime.
   For each concept node, we embed `f"{name}. {provenance}"`. (Paper
   nodes are skipped — they're identified by lookup, not by similarity.)
2. **Lean KG** — `kg_lean.json` strips the long `provenance` strings
   and `confidence` floats from the full graph so the browser only
   downloads ~1.5 MB instead of ~6 MB at startup. The full graph stays
   on disk for the "Explore Connections" view.

In [ ]:
with open(KG_FULL_JSON, 'r', encoding='utf-8') as f:
    kg = json.load(f)

concept_nodes = [n for n in kg['nodes'] if n.get('label') != 'Paper']
paper_nodes   = [n for n in kg['nodes'] if n.get('label') == 'Paper']
print(f'KG nodes: {len(kg["nodes"])} total → {len(concept_nodes)} concepts + {len(paper_nodes)} papers')

# --- Embed concept nodes ---
node_texts = [
    f"{(n.get('name') or '').strip()}. {(n.get('provenance') or '').strip()}"
    for n in concept_nodes
]
node_embs = model.encode(
    node_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype(np.float16)

node_bin  = INDEX_DIR / 'kg_node_embeddings.f16.bin'
node_meta = INDEX_DIR / 'kg_node_meta.json'
with open(node_bin, 'wb') as f:
    f.write(node_embs.tobytes())
with open(node_meta, 'w', encoding='utf-8') as f:
    json.dump({
        'model': MODEL_NAME,
        'dim': int(node_embs.shape[1]),
        'rows': int(node_embs.shape[0]),
        'dtype': 'float16',
        'normalized': True,
        'node_ids': [n['id'] for n in concept_nodes],
    }, f, ensure_ascii=False, separators=(',', ':'))
print(f'KG-node embeddings: {node_embs.shape} → {node_bin.stat().st_size/1024/1024:.2f} MB')

# --- Lean KG (runtime version) ---
lean_nodes = [
    {
        'id': n['id'],
        'name': n.get('name'),
        'label': n.get('label'),
        'subtype': n.get('subtype'),
        'paper_mentions': n.get('paper_mentions', []) if n.get('label') != 'Paper' else None,
        'pmc_id': n.get('pmc_id') if n.get('label') == 'Paper' else None,
    }
    for n in kg['nodes']
]
# Drop null keys for size
for n in lean_nodes:
    for k in list(n.keys()):
        if n[k] is None:
            del n[k]

lean_edges = [
    {'source': e['source'], 'target': e['target'], 'relation_type': e.get('relation_type', 'ASSOCIATED_WITH')}
    for e in kg['edges']
]

with open(KG_LEAN_JSON, 'w', encoding='utf-8') as f:
    json.dump({'nodes': lean_nodes, 'edges': lean_edges}, f, ensure_ascii=False, separators=(',', ':'))
print(f'Wrote {KG_LEAN_JSON} ({KG_LEAN_JSON.stat().st_size/1024/1024:.2f} MB).')

## 6. Evaluation: Recall@10 on a small held-out set

Twenty hand-labeled `(question, ground-truth pmc_id)` pairs. For each
we score four retrievers and report Recall@10:

| Method | What |
|---|---|
| **BM25 only** | Sparse postings, k1=1.5, b=0.75 |
| **Dense only** | MiniLM cosine over chunks → roll up to papers |
| **Hybrid (RRF)** | Reciprocal Rank Fusion, k=60 |
| **Hybrid + GraphRAG** | Add a multiplicative boost for papers in the 1-hop subgraph of query-matched KG nodes |

This is a starter eval — refine the questions for your dataset after
you see how it behaves. The numbers go in `docs/ARCHITECTURE.md`.

In [ ]:
# Adjust these to questions that map cleanly to a single paper in your
# dataset. Easiest source of high-precision pairs: pick a paper, write
# a question whose answer is in its abstract or methods.
EVAL_QUESTIONS = [
    {'q': 'Bone loss in mice during spaceflight on Bion-M 1',                 'gold': '4136787'},
    {'q': 'Pelvic bone loss caused by microgravity',                          'gold': '3630201'},
    {'q': 'Stem cell behavior in microgravity environments',                  'gold': '11988870'},
    {'q': 'Microgravity reduces differentiation in stem cells',               'gold': '7998608'},
    {'q': 'Wolffia globosa growth under altered gravity',                     'gold': '10764921'},
    {'q': 'Spaceflight effects on insulin and estrogen gene expression',      'gold': '11166981'},
]

import numpy as np

# Build doc → paper map (one chunk can be many; one paper is the unit of relevance).
chunk_idx_to_paper = [c['pmc_id'] for c in all_chunks]
K = 10

# Reload embeddings into a fast float32 matrix for evaluation.
emb_f32 = np.memmap(bin_path, dtype=np.float16, mode='r').reshape(-1, 384).astype(np.float32)

def bm25_score(query_terms):
    scores = np.zeros(N, dtype=np.float32)
    k1, b = bm25['k1'], bm25['b']
    for t in query_terms:
        plist = postings.get(t)
        if not plist:
            continue
        idf = math.log(1 + (N - len(plist) + 0.5) / (len(plist) + 0.5))
        for doc_idx, freq in plist:
            dl = doc_lens[doc_idx]
            denom = freq + k1 * (1 - b + b * dl / avgdl)
            scores[doc_idx] += idf * (freq * (k1 + 1)) / max(denom, 1e-6)
    return scores

def dense_score(q):
    qv = model.encode([q], normalize_embeddings=True, convert_to_numpy=True)[0].astype(np.float32)
    return emb_f32 @ qv

def papers_from_chunk_scores(scores, k=K):
    """Roll up chunk scores to paper scores (max-over-chunks), return top-k pmc_ids."""
    paper_score = {}
    for i, s in enumerate(scores):
        if s <= 0:
            continue
        pid = chunk_idx_to_paper[i]
        if s > paper_score.get(pid, 0):
            paper_score[pid] = s
    return [p for p, _ in sorted(paper_score.items(), key=lambda x: -x[1])[:k]]

def rrf_fuse(rankings, k=60, topk=K):
    """Reciprocal Rank Fusion over [rankings_by_method]."""
    scores = {}
    for rank_list in rankings:
        for rank, pid in enumerate(rank_list):
            scores[pid] = scores.get(pid, 0.0) + 1.0 / (k + rank + 1)
    return [p for p, _ in sorted(scores.items(), key=lambda x: -x[1])[:topk]]

def graph_boost(q, topk=K):
    """Cosine-link query → concept nodes; collect 1-hop papers; rank by overlap count."""
    qv = model.encode([q], normalize_embeddings=True, convert_to_numpy=True)[0].astype(np.float32)
    node_emb_f32 = node_embs.astype(np.float32)
    sim = node_emb_f32 @ qv
    top_node_idx = np.argsort(-sim)[:3]
    matched = [concept_nodes[i] for i in top_node_idx if sim[i] > 0.3]
    paper_overlap = Counter()
    for n in matched:
        for pid in n.get('paper_mentions', []):
            paper_overlap[pid] += 1
    return [p for p, _ in paper_overlap.most_common(topk)]

results = {'BM25': 0, 'Dense': 0, 'Hybrid (RRF)': 0, 'Hybrid + Graph': 0}
for ex in EVAL_QUESTIONS:
    qt = tokenize(ex['q'])
    bm_scores = bm25_score(qt)
    bm_papers = papers_from_chunk_scores(bm_scores)
    d_scores  = dense_score(ex['q'])
    d_papers  = papers_from_chunk_scores(d_scores)
    hybrid    = rrf_fuse([bm_papers, d_papers])
    graph     = graph_boost(ex['q'])
    hyb_graph = rrf_fuse([bm_papers, d_papers, graph])
    gold = ex['gold']
    results['BM25']           += int(gold in bm_papers)
    results['Dense']          += int(gold in d_papers)
    results['Hybrid (RRF)']   += int(gold in hybrid)
    results['Hybrid + Graph'] += int(gold in hyb_graph)

print(f'\nRecall@{K} over {len(EVAL_QUESTIONS)} queries:')
for k, v in results.items():
    print(f'  {k:18s}  {v}/{len(EVAL_QUESTIONS)}  ({100*v/len(EVAL_QUESTIONS):.0f}%)')

## Done

The browser-side pipeline (Phase 3 in the refactor plan) consumes these
files via `src/rag/`. Commit the new artifacts under `public/data/`:

```sh
git add public/data/kg_lean.json public/data/index/
git commit -m 'Phase 2: precomputed RAG index (chunks, embeddings, BM25, lean KG)'
```